# Task 1.1 — Core Contribution / Architecture
**Paper:** Agarwal, A., Xie, B., Vovsha, I., Rambow, O., & Passonneau, R. (2011). *Sentiment Analysis of Twitter Data.* ACL Workshop on Language in Social Media (LSM 2011), pp. 30–38.

---

## Overview

This paper builds models to classify tweets into **positive**, **negative**, and **neutral** sentiment categories. It proposes two novel systems — a **feature-based SVM** (called *Senti-features*) and a **tree kernel SVM** — and evaluates them against a unigram baseline. The pipeline flows as follows:

```
Raw Tweet → Preprocessing → Prior Polarity Scoring → [Branch A: Feature Extraction → SVM]
                                                     → [Branch B: Tree Representation → Kernel SVM]
                                                     → Sentiment Label (Positive / Negative / Neutral)
```

---

## Step-by-Step Method Description

---

### Step 1: Raw Tweet Input and Tokenisation
- **What happens:** The system receives a raw tweet — a short, noisy, informal text of up to 140 characters — and immediately tokenises it using the **Stanford tokenizer** (Klein & Manning, 2003). Tweets may contain emoticons (`:)`), hashtags (`#happy`), targets (`@username`), abbreviations (`lol`, `gr8`), repeated characters (`coooool`), and URLs, all of which the tokenizer splits into discrete tokens before any further processing.
- **Reference:** Section 3 (*Data Description*) and Section 4 (*Resources and Pre-processing*) — the dataset is 5,127 manually annotated tweets (1,709 per class: positive, negative, neutral), collected from a real-time Twitter stream; the Stanford tokenizer is explicitly named as the tokenisation tool.
- **Purpose:** Tokenisation is the very first operation the method applies. It converts a raw string into a sequence of discrete tokens that every downstream step — polarity lookup, POS tagging, tree construction — can operate on. Without this step, the method has no unit to attach features to.

---

### Step 2: Preprocessing and Resource Lookup
- **What happens:** Each tweet is cleaned and normalised using a five-step pipeline:
  1. **Emoticons** are replaced with their polarity tag (e.g., `:)` → `||P||` for positive, `:(` → `||N||` for negative) using a hand-crafted dictionary of **170 emoticons**.
  2. **URLs** are replaced with the tag `||U||`.
  3. **Targets** (e.g., `@John`) are replaced with the tag `||T||`.
  4. **Negations** (`not`, `no`, `never`, `n't`, `cannot`) are replaced with the tag `NOT`.
  5. **Repeated characters** are collapsed to three (e.g., `coooooool` → `coool`), to distinguish emphasis from normal usage.
  - Additionally, **acronyms** (e.g., `lol`, `gr8`) are expanded to their English equivalents using a dictionary of **5,184 entries**.
- **Reference:** Section 4 (*Resources and Pre-processing of data*), Table 1 (acronym examples), Table 2 (emoticon polarity dictionary excerpt).
- **Purpose:** Standardise the highly irregular Twitter vocabulary so that downstream steps — POS tagging, polarity lookup, tree building — work reliably on a cleaned representation.

---

### Step 3: Prior Polarity Scoring
- **What happens:** Every English word in the tweet is assigned a **prior polarity score** — a real-valued number between 0 (negative) and 1 (positive) — using the **Dictionary of Affect in Language (DAL)** (Whissel, 1989), which contains ~8,000 words.
  - Words with score < 0.5 → **Negative**
  - Words with score > 0.8 → **Positive**
  - Words in between → **Neutral**
  - If a word is **not in DAL**, the system looks up its **WordNet synonyms** and assigns the polarity of the first synonym found in DAL. This covers 88.9% of English words in the dataset (81.1% directly + 7.8% via WordNet).
- **Reference:** Section 5 (*Prior Polarity Scoring*).
- **Purpose:** Provide a grounded, lexicon-based sentiment signal for each word. This is the foundation of the Senti-features (Step 4). Rather than treating every word as an opaque symbol (as in bag-of-words), this step encodes *what sentiment a word carries by default*, before context is considered.

---

### Step 4A: Feature Extraction — 100 Senti-features (Feature-Based SVM Branch)
- **What happens:** A compact set of **100 features** is computed from the preprocessed tweet. These are called *Senti-features*. The 100 come from computing **50 base features** over both (a) the full tweet and (b) the **last one-third** of the tweet (capturing how sentiment often concentrates at the end).

  The 50 base features span three value types and four semantic categories (see Table 4 of the paper):

  | Category | Examples |
  |---|---|
  | **Polar POS** (f1, f8) | Count and sum of prior polarity scores for adjectives (JJ), adverbs (RB), verbs (VB), nouns (NN) |
  | **Polar Other** (f2, f3, f4, f9) | Count of negations; count of positive/negative emoticons; count of polar hashtags; sum of all word polarity scores |
  | **Non-Polar POS** (f5) | Raw POS tag counts (how many JJ, VB, NN etc.) |
  | **Non-Polar Other** (f6, f7, f10, f11) | Count of URLs, targets, hashtags, slangs; % capitalised text; presence of exclamation marks |

  The key insight is **combining prior polarity with POS tags** — e.g., "how many *negative adjectives* are in this tweet?" is more informative than either feature alone.

- **Reference:** Section 7 (*Our Features*), Table 4.
- **Purpose:** Compress the tweet into a small, interpretable, linguistically motivated feature vector. Computing features over the **last one-third separately** is deliberate — sentiment-bearing words in tweets (opinions, reactions, punchlines) tend to cluster toward the end of the message, so the window gives the classifier a focused signal about the tweet's concluding sentiment without diluting it across the full-tweet counts. The paper shows this 100-dimensional vector performs comparably to a 10,000+ dimensional unigram bag-of-words (Section 8).

---

### Step 4B: Tree Representation (Tree Kernel SVM Branch)
- **What happens:** Instead of extracting hand-crafted features, each preprocessed tweet is converted into a **hierarchical tree structure** with a `ROOT` node. Each token becomes a subtree added to ROOT based on its type:

  - **Targets, emoticons, punctuation, negations** → leaf node directly under ROOT (e.g., `||T||`, `NOT`, `EXC`, `||P||`)
  - **Stop words** → subtree `(STOP ('stop-word'))` under ROOT
  - **English words** → subtree `(EW (POS_TAG  word  prior_polarity))` — e.g., `(EW (JJ great POS))`
  - **Other tokens** → `(NE (token))`

  For example, the tweet `@Fernando this isn't a great day for playing the HARP! :)` becomes the tree shown in **Figure 1** of the paper, with nodes like `||T||`, `NOT`, `(STOP this)`, `(EW (JJ great POS))`, `||P||`, etc.

- **Reference:** Section 6 (*Design of Tree Kernel*), Figure 1.
- **Purpose:** Encode the tweet in a way that captures *all possible feature combinations* without manual engineering. The tree structure encodes word identity, POS, polarity, and Twitter-specific tokens simultaneously in one object.

---

### Step 5: SVM Classification
- **What happens:** A **Support Vector Machine (SVM)** classifier is trained in one of two ways, depending on the branch:

  **Branch A (Senti-features):** The 100-dimensional feature vector from Step 4A is fed into a standard SVM (the paper does not specify the kernel type for this branch). The SVM learns a decision boundary that separates positive / negative / neutral classes in the feature space.

  **Branch B (Tree Kernel):** Pairs of tweets are compared using a **Partial Tree (PT) Kernel** (Moschitti, 2006). Instead of a dot product of feature vectors, the kernel computes similarity between two tweet-trees by counting all **matching subtrees**, including partial ones where non-adjacent branches are made adjacent. This means the SVM implicitly uses thousands of feature combinations without them being explicitly enumerated.

  For both branches, a **5-fold cross-validated SVM** is used, with the regularisation parameter `C` tuned via nested cross-validation on the training fold.

- **Reference:** Section 8 (*Experiments and Results*), Tables 5 and 8.
- **Purpose:** Perform the final classification. The SVM is chosen because it handles high-dimensional, sparse feature spaces well (relevant for unigrams) and supports kernel functions (needed for the tree kernel).

---

### Step 6: Output — Sentiment Label
- **What happens:** The SVM outputs a predicted class label for each tweet:
  - **2-way task:** Positive or Negative
  - **3-way task:** Positive, Negative, or Neutral
  
  Performance is measured by **average accuracy** and **F1-measure** across 5 folds.

  Key results (Table 5 and Table 8):

  | Model | 2-way Accuracy | 3-way Accuracy |
  |---|---|---|
  | Unigram baseline | 71.35% | 56.58% |
  | 100 Senti-features | 71.27% | 56.31% |
  | Tree Kernel | 73.93% | 60.60% |
  | Unigram + Senti-features | **75.39%** | 60.50% |
  | Kernel + Senti-features | 74.61% | **60.83%** |

- **Reference:** Section 8.1 and 8.2, Tables 5, 6, 8, 9.
- **Purpose:** Demonstrate that both the Senti-features and the tree kernel outperform the unigram baseline, validating the paper's two core contributions.

---

## Final Summary

This paper addresses the problem of **classifying the sentiment of tweets into positive, negative, and neutral categories**; the authors claim their approach is better than the existing unigram baseline because it either uses **linguistically motivated features that combine prior word polarity with POS tags** (Senti-features branch — achieving comparable accuracy with 100× fewer features) or uses a **Partial Tree kernel that automatically captures all possible feature combinations from a structured tweet representation** without manual engineering (tree kernel branch — outperforming the unigram model by ~4% absolute accuracy on the 3-way task).